# Sentix Tinder - Phase 4: BERT Fine-Tuning v2 (Binary Sentiment)
Run this notebook on Kaggle or Google Colab with **GPU Acceleration** enabled.

## 1. Install & Import Dependencies

In [ ]:
!pip install transformers datasets scikit-learn onnx onnxruntime accelerate emoji
import pandas as pd
import numpy as np
import torch
import re
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import os


## 2. Load Raw Dataset & Minimal Preprocessing
**IMPORTANT:** Please upload `raw_reviews.csv` to your Kaggle Notebook/Colab environment.

In [ ]:
data_path = "./raw_reviews.csv"
if not os.path.exists(data_path):
    data_path = "/kaggle/input/raw-reviews/raw_reviews.csv" # Or Colab Content directory

df = pd.read_csv(data_path)

def get_sentiment(r):
    try:
        r = float(r)
        if r <= 2: return 1 # Negative
        elif r >= 4: return 0 # Positive
        else: return None # Drop Neutral
    except (ValueError, TypeError):
        return None

df["label"] = df["star_rating"].apply(get_sentiment)
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

def minimal_clean(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    try:
        import emoji
        text = emoji.demojize(text, delimiters=(" ", " "))
    except ImportError:
        text = re.sub(r"[^\x00-\x7F]+", " ", text)
    text = re.sub(r"[^\w\s!?.,']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["review_text"].apply(minimal_clean)
df = df[df["text"].str.split().str.len() >= 4].reset_index(drop=True)

print(f"Dataset size after filtering Neutral and cleaning: {len(df)}")
print(f"\nSample cleaned texts:")
for text in df["text"].sample(5, random_state=42):
    print(f"  -> {text}")

train_df, test_df = train_test_split(df[["text", "label"]], test_size=0.2, random_state=42, stratify=df["label"])
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print(f"Train size: {len(train_dataset)}  Test size: {len(test_dataset)}")


## 3. Tokenize & Train

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions), "f1_macro": f1_score(labels, predictions, average="macro")}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()
results = trainer.evaluate()
print(f"Final Results: {results}")


## 4. Export to ONNX
Run this cell to generate `model.onnx`.

In [ ]:
model.eval()
model.to("cpu")
dummy_text = "This is a test review."
inputs = tokenizer(dummy_text, return_tensors="pt")

onnx_path = "model.onnx"
input_names = ["input_ids", "attention_mask"] + (["token_type_ids"] if "token_type_ids" in inputs else [])
dummy_inputs = tuple(inputs[k] for k in input_names)

import torch.onnx
torch.onnx.export(
    model, 
    args=dummy_inputs, 
    f=onnx_path, 
    input_names=input_names, 
    output_names=["logits"], 
    dynamic_axes={k: {0: "batch_size", 1: "sequence_length"} for k in input_names} | {"logits": {0: "batch_size"}},
    opset_version=14
)
print(f"Successfully exported to {onnx_path}! Please download this file.")
